In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
# Same settings as benchmark-sft-model.ipynb so SFT vs DPO scores are comparable
load_in_4bit = True

In [ ]:
model_finetuned = 'FrancescoArno94/Qwen3-1.7B-dpo-ultrafeedback_binarized-16bit'
trained_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_finetuned,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
FastLanguageModel.for_inference(trained_model)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-3",
)

# Test inference

In [ ]:
def invoke_model(input_query, model, max_new_tokens=64, temperature=0.7, min_p=0.1):
    messages = [
        {
            "role": "user", 
             "content": input_query
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
        return_dict=True
    ).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                             temperature = temperature, min_p = min_p)
    output_list = tokenizer.batch_decode(outputs)
    return output_list[0]

In [ ]:
msg= "Resolve this equation: 2x+1=0"

print(invoke_model(msg, trained_model))

In [ ]:
msg= "A python function that split a string"
print(invoke_model(msg, trained_model, max_new_tokens=256, temperature=0.1))

# Benchmark (gsm8k, mmlu, truthfulqa_mc2)

In [ ]:
!pip install lm-eval

In [ ]:
import lm_eval
from lm_eval.loggers import EvaluationTracker
from lm_eval.models.huggingface import HFLM

tracker = EvaluationTracker(output_path="results")
lm = HFLM(pretrained=trained_model, tokenizer=tokenizer, batch_size=8)
res = lm_eval.simple_evaluate(model=lm, tasks=["gsm8k", "mmlu", "truthfulqa_mc2"], apply_chat_template=True,
                              evaluation_tracker=tracker)
print(res["results"])

In [ ]:
tracker.save_results_aggregated(results=res, samples=res["samples"])
for task in res["configs"]:
    tracker.save_results_samples(task_name=task, samples=res["samples"][task])

print("Results saved")